In [1]:
import sys
from pathlib import Path

sys.path.append(
    str(Path.cwd().parent)
)

In [2]:
from src.governance.model_registry import (
    ModelRegistry
)

In [3]:
registry = ModelRegistry()

In [4]:
registry.register_model(
    model_id="PD_RETAIL_001",
    model_name="Retail Probability of Default Model",
    model_type="Logistic Regression",
    version="1.0",
    owner="Credit Risk",
    risk_type="Credit Risk"
)

In [5]:
registry.register_model(
    model_id="LGD_RETAIL_001",
    model_name="Retail Loss Given Default Model",
    model_type="Random Forest Regression",
    version="1.0",
    owner="Credit Risk",
    risk_type="Credit Risk"
)

In [6]:
registry.register_model(
    model_id="EAD_RETAIL_001",
    model_name="Retail Exposure at Default Model",
    model_type="Random Forest Regression",
    version="1.0",
    owner="Credit Risk",
    risk_type="Credit Risk"
)

In [9]:
import pandas as pd
model_registry_df = pd.DataFrame(
    models
)

model_registry_df

,model_id,model_name,model_type,version,owner,risk_type,status,registered_at
0,PD_RETAIL_001,Retail Probability of Default Model,Logistic Regression,1.0,Credit Risk,Credit Risk,ACTIVE,2026-08-11T02:58:47.028324
1,LGD_RETAIL_001,Retail Loss Given Default Model,Random Forest Regression,1.0,Credit Risk,Credit Risk,ACTIVE,2026-08-11T02:59:09.361650
2,EAD_RETAIL_001,Retail Exposure at Default Model,Random Forest Regression,1.0,Credit Risk,Credit Risk,ACTIVE,2026-08-11T02:59:18.647605


In [10]:
model_registry_df.to_csv(
    "../data/outputs/model_registry.csv",
    index=False
)

In [11]:
from src.governance.findings import (
    FindingManager
)

In [12]:
finding_manager = FindingManager()

In [15]:
from src.validation.stability import calculate_psi

In [16]:
import inspect

print(inspect.signature(calculate_psi))

(expected, actual, bins=10)


In [20]:
df = pd.read_csv(
    "../data/raw/credit_portfolio.csv"
)

In [22]:
df["observation_date"] = pd.to_datetime(
    df["observation_date"]
)

In [23]:
df[
    "observation_date"
].min(), df[
    "observation_date"
].max()

(Timestamp('2022-01-01 00:00:00'), Timestamp('2023-12-01 00:00:00'))

In [24]:
baseline_data = df[
    df["observation_date"] < "2023-07-01"
].copy()
print(
    "Baseline rows:",
    len(baseline_data)
)

Baseline rows: 180000


In [26]:
oot_data = df[
    df["observation_date"] >= "2023-10-01"
].copy()

OOT rows: 30000


In [29]:
baseline_data[
    ["observation_date", "credit_score"]
].head()

,observation_date,credit_score
0,2022-01-01,680.0
1,2022-02-01,700.0
2,2022-03-01,692.0
3,2022-04-01,688.0
4,2022-05-01,545.0


In [30]:
oot_data[
    ["observation_date", "credit_score"]
].head()

,observation_date,credit_score
21,2023-10-01,665.0
22,2023-11-01,702.0
23,2023-12-01,621.0
45,2023-10-01,736.0
46,2023-11-01,752.0


In [32]:
credit_score_psi = calculate_psi(
    expected=baseline_data["credit_score"],
    actual=oot_data["credit_score"]
)
print(
    f"Credit Score PSI: {credit_score_psi:.4f}"
)

Credit Score PSI: 0.0006


In [33]:
expected=baseline_data["credit_score"]
actual=oot_data["credit_score"]

In [35]:
from src.monitoring.thresholds import classify_psi
psi_status = classify_psi(
    credit_score_psi
)

In [36]:
print(
    "Credit Score PSI Status:",
    psi_status
)

Credit Score PSI Status: GREEN


In [37]:
if credit_score_psi > 0.25:

    finding_manager.create_finding(
        finding_id="MRM-001",
        model_id="PD_RETAIL_001",
        finding_type="Stability",
        severity="HIGH",
        metric="PSI",
        observed_value=credit_score_psi,
        threshold=0.25,
        description=(
            "Credit score PSI exceeded "
            "the defined monitoring threshold."
        ),
        recommendation=(
            "Investigate population drift, "
            "perform root-cause analysis, "
            "and assess whether recalibration "
            "or redevelopment is required."
        )
    )

In [38]:
monitoring_variables = [
    "credit_score",
    "annual_income",
    "debt_to_income",
    "credit_utilization",
    "loan_amount",
    "interest_rate"
]

In [39]:
psi_results = []

for variable in monitoring_variables:

    psi = calculate_psi(
        expected=baseline_data[variable],
        actual=oot_data[variable]
    )

    psi_results.append({
        "variable": variable,
        "psi": psi,
        "status": classify_psi(psi)
    })

In [40]:
psi_results_df = pd.DataFrame(
    psi_results
)

In [41]:
psi_results_df

,variable,psi,status
0,credit_score,0.000639,GREEN
1,annual_income,0.000153,GREEN
2,debt_to_income,0.000403,GREEN
3,credit_utilization,0.000163,GREEN
4,loan_amount,0.000451,GREEN
5,interest_rate,0.000320,GREEN


In [42]:
psi_results_df.to_csv(
    "../data/outputs/psi_results.csv",
    index=False
)

In [43]:
for _, row in psi_results_df.iterrows():

    if row["status"] == "RED":

        finding_manager.create_finding(
            finding_id=(
                f"PSI-{row['variable']}"
            ),
            model_id="PD_RETAIL_001",
            finding_type="Stability",
            severity="HIGH",
            metric="PSI",
            observed_value=row["psi"],
            threshold=0.25,
            description=(
                f"PSI for {row['variable']} "
                "exceeded the defined "
                "monitoring threshold."
            ),
            recommendation=(
                "Investigate population drift, "
                "perform root-cause analysis, "
                "and assess whether "
                "recalibration or redevelopment "
                "is required."
            )
        )

In [44]:
findings_df = pd.DataFrame(
    finding_manager.get_findings()
)

findings_df

""


In [47]:
from src.validation.calibration import calibration_table

print(dir(calibration))

['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'calibration_table', 'pd']


In [52]:
import joblib

logistic_model = joblib.load(
    "../models/pd_model.joblib"
)


['age' 'annual_income' 'employment_years' 'credit_score' 'debt_to_income'
 'credit_utilization' 'previous_defaults' 'delinquencies_12m'
 'loan_amount' 'loan_term_months' 'interest_rate' 'unemployment_rate'
 'gdp_growth']


In [106]:
pd_features = list(
    logistic_model.feature_names_in_
)

X_oot_pd = oot_data[pd_features]
print(pd_features)

['age', 'annual_income', 'employment_years', 'credit_score', 'debt_to_income', 'credit_utilization', 'previous_defaults', 'delinquencies_12m', 'loan_amount', 'loan_term_months', 'interest_rate', 'unemployment_rate', 'gdp_growth']


In [54]:
predicted_pd = logistic_model.predict_proba(
    X_oot_pd
)[:, 1]

In [56]:
y_true = oot_data["default_flag"]

In [60]:
calibration = calibration_table(
    y_true=y_true,
    predicted_pd=predicted_pd,
    n_buckets=10
)
calibration

,bucket,predicted_pd,realized_default_rate,observations,calibration_error
0,0,0.180499,0.023333,3000,-0.157165
1,1,0.259786,0.046000,3000,-0.213786
2,2,0.312488,0.053000,3000,-0.259488
3,3,0.359903,0.073667,3000,-0.286237
4,4,0.406519,0.074000,3000,-0.332519
5,5,0.453037,0.092333,3000,-0.360703
6,6,0.502395,0.110000,3000,-0.392395
7,7,0.558473,0.136333,3000,-0.422140
8,8,0.627279,0.171667,3000,-0.455612
9,9,0.742789,0.274667,3000,-0.468123


In [61]:
max_calibration_error = (
    calibration[
        "calibration_error"
    ]
    .abs()
    .max()
)

In [62]:
print(
    f"Maximum calibration error: "
    f"{max_calibration_error:.4f}"
)

Maximum calibration error: 0.4681


In [63]:
if max_calibration_error > 0.03:
    print(
        "CALIBRATION BREACH"
    )
else:
    print(
        "CALIBRATION WITHIN THRESHOLD"
    )

CALIBRATION BREACH


In [64]:
if max_calibration_error > 0.03:

    finding_manager.create_finding(
        finding_id="MRM-002",
        model_id="PD_RETAIL_001",
        finding_type="Calibration",
        severity="MEDIUM",
        metric="Calibration Error",
        observed_value=max_calibration_error,
        threshold=0.03,
        description=(
            "Observed calibration error "
            "exceeded the monitoring threshold."
        ),
        recommendation=(
            "Investigate systematic "
            "overprediction or underprediction "
            "and assess recalibration."
        )
    )

In [65]:
findings_df = pd.DataFrame(
    finding_manager.get_findings()
)

findings_df

,finding_id,model_id,finding_type,severity,metric,observed_value,threshold,description,recommendation,status,created_at
0,MRM-002,PD_RETAIL_001,Calibration,MEDIUM,Calibration Error,0.468123,0.03,Observed calibration error exceeded the monito...,Investigate systematic overprediction or under...,OPEN,2026-08-11T03:31:46.203372


In [66]:
findings_df = pd.DataFrame(
    finding_manager.get_findings()
)

findings_df

,finding_id,model_id,finding_type,severity,metric,observed_value,threshold,description,recommendation,status,created_at
0,MRM-002,PD_RETAIL_001,Calibration,MEDIUM,Calibration Error,0.468123,0.03,Observed calibration error exceeded the monito...,Investigate systematic overprediction or under...,OPEN,2026-08-11T03:31:46.203372


In [67]:
finding_manager.update_status(
    "MRM-001",
    "UNDER INVESTIGATION"
)

In [68]:
finding_manager.get_findings()

[{'finding_id': 'MRM-002',
  'model_id': 'PD_RETAIL_001',
  'finding_type': 'Calibration',
  'severity': 'MEDIUM',
  'metric': 'Calibration Error',
  'observed_value': np.float64(0.46812272032046914),
  'threshold': 0.03,
  'description': 'Observed calibration error exceeded the monitoring threshold.',
  'recommendation': 'Investigate systematic overprediction or underprediction and assess recalibration.',
  'status': 'OPEN',
  'created_at': '2026-08-11T03:31:46.203372'}]

In [69]:
finding_manager.update_status(
    "MRM-001",
    "ACTION DEFINED"
)

In [70]:
finding_manager.update_status(
    "MRM-001",
    "IMPLEMENTED"
)

In [72]:
finding_manager.update_status(
    "MRM-001",
    "IMPLEMENTED"
)

In [73]:
finding_manager.update_status(
    "MRM-001",
    "CLOSED"
)

In [74]:
findings_df = pd.DataFrame(
    finding_manager.get_findings()
)

findings_df.to_csv(
    "../data/outputs/findings.csv",
    index=False
)

In [78]:
from src.reporting.report_generator import (
    generate_validation_report
)

In [108]:
import pandas as pd

validation_metrics_df = pd.read_csv(
    "../data/outputs/validation_metrics.csv"
)

validation_metrics_df

,metric,value
0,AUC,0.698481
1,KS,0.289243
2,Gini,0.396962
3,Accuracy Ratio,0.396962


In [109]:
validation_metrics = dict(
    zip(
        validation_metrics_df["metric"],
        validation_metrics_df["value"]
    )
)
validation_metrics

{'AUC': 0.698480910343504,
 'KS': 0.289242580381955,
 'Gini': 0.396961820687008,
 'Accuracy Ratio': 0.396961820687008}

In [119]:
import yaml

with open(
    "../config/scenarios.yaml",
    "r"
) as file:
    config = yaml.safe_load(file)

scenarios = config["scenarios"]

print(scenarios)

{'baseline': {'unemployment_rate': 0, 'gdp_growth': 0, 'interest_rate': 0}, 'adverse': {'unemployment_rate': 2.0, 'gdp_growth': -2, 'interest_rate': 1}, 'severe': {'unemployment_rate': 4, 'gdp_growth': -4, 'interest_rate': 3}}


In [121]:
stress_results = run_stress_test(
    model=logistic_model,
    portfolio=oot,
    features=pd_features,
    scenarios=scenarios
)

In [122]:
stress_results

,scenario,baseline_pd,stressed_pd,pd_change,pd_change_pct
0,Baseline,0.440317,0.440317,0.000000,0.000000
1,baseline,0.440317,0.440317,0.000000,0.000000
2,adverse,0.440317,0.519854,0.079537,18.063672
3,severe,0.440317,0.610428,0.170111,38.633731


In [123]:
missing_features = [
    feature
    for feature in pd_features
    if feature not in oot.columns
]

print("Missing features:", missing_features)

Missing features: []


In [124]:
stress_results.to_csv(
    "../data/outputs/stress_results.csv",
    index=False
)

In [125]:
print(
    stress_results[
        [
            "scenario",
            "baseline_pd",
            "stressed_pd",
            "pd_change",
            "pd_change_pct"
        ]
    ]
)

   scenario  baseline_pd  stressed_pd  pd_change  pd_change_pct
0  Baseline     0.440317     0.440317   0.000000       0.000000
1  baseline     0.440317     0.440317   0.000000       0.000000
2   adverse     0.440317     0.519854   0.079537      18.063672
3    severe     0.440317     0.610428   0.170111      38.633731


In [126]:
for _, row in stress_results.iterrows():

    print(
        f"{row['scenario']}: "
        f"PD changes from "
        f"{row['baseline_pd']:.2%} "
        f"to "
        f"{row['stressed_pd']:.2%} "
        f"({row['pd_change_pct']:.2f}% relative change)"
    )

Baseline: PD changes from 44.03% to 44.03% (0.00% relative change)
baseline: PD changes from 44.03% to 44.03% (0.00% relative change)
adverse: PD changes from 44.03% to 51.99% (18.06% relative change)
severe: PD changes from 44.03% to 61.04% (38.63% relative change)


In [127]:
def classify_stress(pd_change_pct):

    if pd_change_pct < 5:
        return "GREEN"

    elif pd_change_pct < 10:
        return "AMBER"

    else:
        return "RED"

In [128]:
stress_results["status"] = (
    stress_results["pd_change_pct"]
    .apply(classify_stress)
)

In [129]:
stress_results

,scenario,baseline_pd,stressed_pd,pd_change,pd_change_pct,status
0,Baseline,0.440317,0.440317,0.000000,0.000000,GREEN
1,baseline,0.440317,0.440317,0.000000,0.000000,GREEN
2,adverse,0.440317,0.519854,0.079537,18.063672,RED
3,severe,0.440317,0.610428,0.170111,38.633731,RED


In [130]:
def stress_action(status):

    if status == "GREEN":
        return "No action required"

    elif status == "AMBER":
        return (
            "Increase monitoring and "
            "investigate deterioration"
        )

    else:
        return (
            "Perform detailed model review "
            "and assess remediation"
        )

In [131]:
stress_results["recommended_action"] = (
    stress_results["status"]
    .apply(stress_action)
)

In [132]:
stress_results.to_csv(
    "../data/outputs/stress_results.csv",
    index=False
)

In [133]:
findings_df = pd.read_csv(
    "../data/outputs/"
    "findings.csv"
)

In [134]:
generate_validation_report(
    output_path=(
        "../data/outputs/"
        "RiskGuard_Validation_Report.pdf"
    ),
    model_name=(
        "Retail Probability of "
        "Default Model"
    ),
    model_version="1.0",
    validation_metrics=validation_metrics,
    stress_results=stress_results,
    findings=findings
)

In [137]:
calibration.to_csv(
    "../data/outputs/"
    "calibration_results.csv",
    index=False
)